In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-utils
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato utilizzando i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-addon-utils~=0.3.0
```
</details>

Il pacchetto delle utilità per gli addon di Qiskit è una raccolta di funzionalità per integrare i flussi di lavoro che coinvolgono uno o più addon di Qiskit. Ad esempio, questo pacchetto contiene funzioni per la creazione di Hamiltoniani, la generazione di circuiti di evoluzione temporale di Trotter, e il taglio e la combinazione di circuiti quantistici.

## Installazione

Esistono due modi per installare le utilità per gli addon di Qiskit: PyPI e la compilazione dal sorgente. Si raccomanda di installare questi pacchetti in un [ambiente virtuale](https://docs.python.org/3.10/tutorial/venv.html) per garantire la separazione tra le dipendenze dei pacchetti.

### Installazione da PyPI

Il modo più diretto per installare il pacchetto delle utilità per gli addon di Qiskit è tramite PyPI.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit import QuantumCircuit
from qiskit_addon_utils.coloring import auto_color_edges
from qiskit_addon_utils.slicing import combine_slices, slice_by_depth
from collections import defaultdict

backend = FakeSherbrooke()
coupling_map = backend.coupling_map

# Create naive circuit
circuit = QuantumCircuit(backend.num_qubits)
for edge in coupling_map.graph.edge_list():
    circuit.cz(edge[0], edge[1])


# Color the edges of the coupling map
coloring = auto_color_edges(coupling_map)
circuit_with_coloring = QuantumCircuit(backend.num_qubits)

# Make a reverse coloring dict in order to make the circuit
color_to_edge = defaultdict(list)
for edge, color in coloring.items():
    color_to_edge[color].append(edge)

# Place edges in order of color
for edges in color_to_edge.values():
    for edge in edges:
        circuit_with_coloring.cz(edge[0], edge[1])

print(f"The circuit without using edge coloring has depth: {circuit.depth()}")
print(
    f"The circuit using edge coloring has depth: {circuit_with_coloring.depth()}"
)

The circuit without using edge coloring has depth: 37
The circuit using edge coloring has depth: 3


### Installazione dal sorgente
<details>
<summary>
Clicca qui per leggere come installare questo pacchetto manualmente.
</summary>

Se desideri contribuire a questo pacchetto o vuoi installarlo manualmente, prima clona il repository:

In [2]:
import numpy as np
from qiskit import QuantumCircuit

num_qubits = 9
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg" alt="Output of the previous code cell" />

e installa il pacchetto tramite `pip`. Se prevedi di eseguire i tutorial presenti nel repository del pacchetto, installa anche le dipendenze per i notebook. Se prevedi di sviluppare nel repository, installa le dipendenze `dev`.

In [3]:
# Slice circuit into partitions of depth 1
slices = slice_by_depth(qc, 1)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/6d824d97-edc6-4c88-bcfa-428731393f83-0.svg" alt="Output of the previous code cell" />

</details>

## Inizia a usare le utilità
Ci sono diversi moduli all'interno del pacchetto `qiskit-addon-utils`, tra cui uno per la generazione di problemi per la simulazione di sistemi quantistici, la colorazione di grafi per posizionare i gate in un circuito quantistico in modo più efficiente, e il taglio di circuiti, che può essere utile per la [retropropagazione degli operatori](/guides/qiskit-addons-obp). Le sezioni seguenti riassumono ciascun modulo. Anche la [documentazione API](https://docs.quantum.ibm.com/api/qiskit-addon-utils) del pacchetto contiene informazioni utili.

### Generazione di problemi
Il contenuto del modulo [`qiskit_addon_utils.problem_generators`](../api/qiskit-addon-utils/problem-generators) comprende:

- Una funzione [`generate_xyz_hamiltonian()`](../api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian), che genera una rappresentazione `SparsePauliOp` consapevole della connettività per un modello XYZ di tipo Ising:

$$H = \sum_{(j,k)\in E} \left(J_x X_jX_k + J_yY_jY_k + J_zZ_jZ_k\right) + \sum_{j\in V} \left(h_x X_j + h_y Y_j + h_z Z_j\right) $$
- Una funzione [`generate_time_evolution_circuit()`](../api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit), che costruisce un circuito che modella l'evoluzione temporale di un dato operatore.
- Tre diversi oggetti [`PauliOrderStrategy`](../api/qiskit-addon-utils/problem-generators#pauliorderstrategy) per enumerare i diversi ordinamenti delle stringhe di Pauli. Questo è particolarmente utile se usato insieme alla colorazione di grafi e può essere impiegato come argomento sia nella funzione `generate_xyz_hamiltonian()` che in `generate_time_evolution_circuit()`.

### Colorazione di grafi
Il modulo [`qiskit_addon_utils.coloring`](../api/qiskit-addon-utils/coloring) viene usato per colorare gli archi di una mappa di accoppiamento e sfruttare questa colorazione per posizionare i gate in un circuito quantistico in modo più efficiente. Lo scopo di questa mappa di accoppiamento con archi colorati è trovare un insieme di colori per gli archi tale che nessun arco dello stesso colore condivida un nodo comune. Per una QPU, questo significa che i gate lungo archi dello stesso colore (connessioni tra qubit) possono essere eseguiti contemporaneamente, rendendo il circuito più veloce.

Come rapido esempio, puoi usare la funzione [`auto_color_edges()`](../api/qiskit-addon-utils/coloring#auto_color_edges) per generare una colorazione degli archi per un circuito naive che esegue un `CZGate` lungo ogni connessione tra qubit. Il frammento di codice seguente utilizza la mappa di accoppiamento del backend `FakeSherbrooke`, crea questo circuito naive, poi usa la funzione `auto_color_edges()` per creare un circuito equivalente più efficiente.

In [4]:
from qiskit_addon_utils.slicing import slice_by_gate_types

slices = slice_by_gate_types(qc)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d011c56e-d741-491a-8867-54952cb7b757-0.svg" alt="Output of the previous code cell" />

If your workload is designed to exploit the physical qubit connectivity of the QPU it will be run on, you can create slices based on edge coloring. The code snippet below will assign a three-coloring to circuit edges and slice the circuit with respect to the edge coloring. (Note: this only affects non-local gates. Single-qubit gates will be sliced by gate type).

In [5]:
from qiskit_addon_utils.slicing import slice_by_coloring

# Assign a color to each set of connected qubits
coloring = {}
for i in range(num_qubits - 1):
    coloring[(i, i + 1)] = i % 3
coloring[(num_qubits - 1, 0)] = 2

# Create a circuit with operations added in order of color
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
edges = [
    edge for color in range(3) for edge in coloring if coloring[edge] == color
]
for edge in edges:
    qc.cx(edge[0], edge[1])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))

# Create slices by edge color
slices = slice_by_coloring(qc, coloring=coloring)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg" alt="Output of the previous code cell" />

### Taglio
Infine, il modulo [`qiskit-addon-utils.slicing`](../api/qiskit-addon-utils/slicing) contiene funzioni e pass del transpiler per lavorare con la creazione di "slice" di circuiti, ovvero partizioni temporali di un [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit) che si estendono su tutti i qubit. Queste slice sono usate principalmente per la [retropropagazione degli operatori](/guides/qiskit-addons-obp-get-started). I quattro metodi principali con cui un circuito può essere tagliato sono: per tipo di gate, per profondità, per colorazione, o tramite istruzioni [`Barrier`](../api/qiskit/circuit#barrier). L'output di queste funzioni di taglio restituisce una lista di oggetti [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit). I circuiti tagliati possono anche essere ricombinati usando la funzione [`combine_slices()`](../api/qiskit-addon-utils/slicing#combine_slices). Leggi il [riferimento API](../api/qiskit-addon-utils/slicing) del modulo per maggiori informazioni.

Di seguito sono riportati alcuni esempi di come creare queste slice utilizzando il seguente circuito:

In [6]:
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qc.barrier()
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.barrier()
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/5040a678-9d5f-4c8b-8a92-7de04c3fcfda-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg)

Nel caso in cui non ci sia un modo chiaro per sfruttare la struttura di un circuito per la retropropagazione degli operatori, puoi partizionare il circuito in slice di una data profondità.

In [7]:
from qiskit_addon_utils.slicing import slice_by_barriers

slices = slice_by_barriers(qc)
slices[0].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/b1106c2f-711c-4d30-b91a-ea05f47598d8-0.svg" alt="Output of the previous code cell" />

In [8]:
slices[1].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/1afd2111-dd88-4e20-a137-f0c975e9d1bb-0.svg" alt="Output of the previous code cell" />

In [9]:
slices[2].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/360ab773-df81-4975-bb19-cd5f34e69b29-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg)

Se hai una strategia di taglio personalizzata, puoi invece inserire delle barriere nel circuito per delimitare i punti di taglio e usare la funzione [`slice_by_barriers`](../api/qiskit-addon-utils/slicing#slice_by_barriers).